In [12]:
import pandas as pd
import numpy as np
import warnings
import os
import time

warnings.filterwarnings('ignore')

OUTPUT_DIR = './data'
FIGURE_DIR = './figures/nb04'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)


In [13]:
# Part 1: Crime Panel Construction

# Aggregate the 8.4M crime records into a community_area × date panel.

# 1.1 Load and prepare crime data


df = pd.read_parquet('./data/chicago_crimes_clean.parquet',
                     columns=['year', 'month', 'day', 'date_only',
                              'community_area', 'primary_type',
                              'arrest', 'domestic'])

print(f"Loaded: {len(df):,} rows")

# Restrict to 2011-2025 
df['date_only'] = pd.to_datetime(df['date_only'])
df = df[(df['year'] >= 2001) & (df['year'] <= 2025)].copy()
print(f"After filtering 2011-2025: {len(df):,} rows")

# Validate community_area
df['community_area'] = pd.to_numeric(df['community_area'], errors='coerce')
n_missing_ca = df['community_area'].isna().sum()
print(f"Missing community_area: {n_missing_ca:,} ({n_missing_ca/len(df)*100:.2f}%)")
df = df.dropna(subset=['community_area']).copy()
df['community_area'] = df['community_area'].astype(int)

# Validate primary_type
print(f"Unique crime types: {df['primary_type'].nunique()}")

# Validate arrest field
if df['arrest'].dtype == object:
    df['arrest'] = df['arrest'].map({'true': True, 'false': False,
                                     'True': True, 'False': False,
                                     True: True, False: False})
df['arrest'] = df['arrest'].astype(bool)

# Validate domestic field
if df['domestic'].dtype == object:
    df['domestic'] = df['domestic'].map({'true': True, 'false': False,
                                         'True': True, 'False': False,
                                         True: True, False: False})
df['domestic'] = df['domestic'].astype(bool)

print(f"\nClean crime data: {len(df):,} rows")
print(f"Date range: {df['date_only'].min()} to {df['date_only'].max()}")
print(f"Community areas: {df['community_area'].nunique()}")

# 1.2 Define crime categories
# Crime type mappings
VIOLENT_TYPES = ['BATTERY', 'ASSAULT', 'HOMICIDE', 'ROBBERY',
                 'CRIM SEXUAL ASSAULT', 'CRIMINAL SEXUAL ASSAULT']

PROPERTY_TYPES = ['THEFT', 'BURGLARY', 'MOTOR VEHICLE THEFT',
                  'CRIMINAL DAMAGE', 'ARSON']

# Verify these types exist in data
all_types = df['primary_type'].unique()
for t in VIOLENT_TYPES + PROPERTY_TYPES:
    if t not in all_types:
        print(f"  WARNING: '{t}' not found in data")

# Create boolean flags for efficient aggregation
df['is_violent'] = df['primary_type'].isin(VIOLENT_TYPES)
df['is_property'] = df['primary_type'].isin(PROPERTY_TYPES)
df['is_theft'] = df['primary_type'] == 'THEFT'
df['is_battery'] = df['primary_type'] == 'BATTERY'
df['is_homicide'] = df['primary_type'] == 'HOMICIDE'
df['is_burglary'] = df['primary_type'] == 'BURGLARY'
df['is_mvt'] = df['primary_type'] == 'MOTOR VEHICLE THEFT'
df['is_narcotics'] = df['primary_type'] == 'NARCOTICS'
df['is_robbery'] = df['primary_type'] == 'ROBBERY'
df['is_assault'] = df['primary_type'] == 'ASSAULT'

print("Crime category flags created.")

# 1.3 Build the complete panel skeleton

# All unique community areas (1-77)
all_ca = sorted(df['community_area'].unique())
print(f"Community areas in data: {len(all_ca)}")

# All dates in range
date_min = df['date_only'].min()
date_max = df['date_only'].max()
all_dates = pd.date_range(start=date_min, end=date_max, freq='D')
print(f"Date range: {date_min.date()} to {date_max.date()} ({len(all_dates)} days)")

# Create complete skeleton
skeleton = pd.MultiIndex.from_product(
    [all_ca, all_dates],
    names=['community_area', 'date']
).to_frame(index=False)

expected_rows = len(all_ca) * len(all_dates)
print(f"Expected panel rows: {len(all_ca)} CA x {len(all_dates)} days = {expected_rows:,}")
print(f"Skeleton rows: {skeleton.shape[0]:,}")
assert skeleton.shape[0] == expected_rows, "Skeleton row count mismatch!"

# 1.4 Aggregate crime data to community_area × date

# Aggregate all Y variables in one pass
agg_dict = {
    'primary_type': 'size',          # total crime count
    'is_violent': 'sum',
    'is_property': 'sum',
    'is_theft': 'sum',
    'is_battery': 'sum',
    'is_homicide': 'sum',
    'is_burglary': 'sum',
    'is_mvt': 'sum',
    'is_narcotics': 'sum',
    'is_robbery': 'sum',
    'is_assault': 'sum',
    'arrest': 'sum',                 # arrest count
    'domestic': 'sum',               # domestic crime count
}

crime_agg = (df.groupby(['community_area', 'date_only'])
             .agg(agg_dict)
             .reset_index())

# Rename columns
crime_agg.columns = ['community_area', 'date',
                      'crime_total', 'crime_violent', 'crime_property',
                      'crime_theft', 'crime_battery', 'crime_homicide',
                      'crime_burglary', 'crime_mvt', 'crime_narcotics',
                      'crime_robbery', 'crime_assault',
                      'arrest_count', 'domestic_count']

print(f"Aggregated crime data: {crime_agg.shape[0]:,} rows")
print(f"(These are only date-CA combinations that had at least 1 crime)")

1.5 Merge skeleton with crime data


panel = skeleton.merge(crime_agg, on=['community_area', 'date'], how='left')

# Fill NaN with 0 (days with no crime)
y_cols = ['crime_total', 'crime_violent', 'crime_property',
          'crime_theft', 'crime_battery', 'crime_homicide',
          'crime_burglary', 'crime_mvt', 'crime_narcotics',
          'crime_robbery', 'crime_assault',
          'arrest_count', 'domestic_count']

for col in y_cols:
    n_null = panel[col].isna().sum()
    panel[col] = panel[col].fillna(0).astype(int)
    if n_null > 0:
        print(f"  {col}: filled {n_null:,} zeros "
              f"({n_null/len(panel)*100:.1f}% of panel)")

# Compute arrest rate (avoid division by zero)
panel['arrest_rate'] = np.where(
    panel['crime_total'] > 0,
    panel['arrest_count'] / panel['crime_total'],
    0.0
)

print(f"\nPanel shape: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
assert panel.shape[0] == expected_rows, "Panel row count changed after merge!"

# 1.6 Validate crime panel

print("=" * 60)
print("Crime Panel Validation")
print("=" * 60)

# Check no missing values
print(f"\nMissing values per column:")
for col in panel.columns:
    n = panel[col].isna().sum()
    status = "OK" if n == 0 else f"PROBLEM: {n:,} missing"
    print(f"  {col:<20s} {status}")

# Check value ranges
print(f"\nValue range checks:")
for col in y_cols:
    vmin, vmax, vmean = panel[col].min(), panel[col].max(), panel[col].mean()
    print(f"  {col:<20s} min={vmin:>5d}  max={vmax:>5d}  mean={vmean:.2f}")

# Check arrest rate
print(f"\n  arrest_rate         min={panel['arrest_rate'].min():.3f}  "
      f"max={panel['arrest_rate'].max():.3f}  "
      f"mean={panel['arrest_rate'].mean():.3f}")

# Spot check: total should match original data
total_from_panel = panel['crime_total'].sum()
total_from_raw = len(df)
print(f"\nTotal crime count from panel: {total_from_panel:,}")
print(f"Total crime count from raw:   {total_from_raw:,}")
assert total_from_panel == total_from_raw, "MISMATCH in total crime count!"
print("PASS: totals match.")

# Spot check: a few specific communities
for ca in [25, 32, 8]:  # Austin, Loop, Near North Side
    ca_total = panel.loc[panel['community_area'] == ca, 'crime_total'].sum()
    ca_raw = len(df[df['community_area'] == ca])
    match = "PASS" if ca_total == ca_raw else "FAIL"
    print(f"  CA {ca}: panel={ca_total:,} raw={ca_raw:,} [{match}]")

Loaded: 8,408,989 rows
After filtering 2011-2025: 8,367,792 rows
Missing community_area: 603,481 (7.21%)
Unique crime types: 33

Clean crime data: 7,764,311 rows
Date range: 2001-01-01 00:00:00 to 2025-12-31 00:00:00
Community areas: 78
Crime category flags created.
Community areas in data: 78
Date range: 2001-01-01 to 2025-12-31 (9131 days)
Expected panel rows: 78 CA x 9131 days = 712,218
Skeleton rows: 712,218
Aggregated crime data: 651,489 rows
(These are only date-CA combinations that had at least 1 crime)
  crime_total: filled 60,729 zeros (8.5% of panel)
  crime_violent: filled 60,729 zeros (8.5% of panel)
  crime_property: filled 60,729 zeros (8.5% of panel)
  crime_theft: filled 60,729 zeros (8.5% of panel)
  crime_battery: filled 60,729 zeros (8.5% of panel)
  crime_homicide: filled 60,729 zeros (8.5% of panel)
  crime_burglary: filled 60,729 zeros (8.5% of panel)
  crime_mvt: filled 60,729 zeros (8.5% of panel)
  crime_narcotics: filled 60,729 zeros (8.5% of panel)
  crime_ro

In [14]:
print(sorted(panel['community_area'].unique()))

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), n

In [15]:
panel = panel[panel['community_area'].between(1, 77)].copy()
print(f"After filtering CA 1-77: {len(panel):,} rows")

After filtering CA 1-77: 703,087 rows


In [16]:
# Part 2: Weather Data
# Daily weather variables from the Open-Meteo Historical Weather API
# for Chicago (41.8781°N, 87.6298°W).

# Weather is city-level (same for all community areas on a given day).
# 2.1 Fetch weather data

WEATHER_FILE = os.path.join(OUTPUT_DIR, 'chicago_weather_daily.csv')

if os.path.exists(WEATHER_FILE):
    print(f"Weather file found: {WEATHER_FILE}")
    weather = pd.read_csv(WEATHER_FILE, parse_dates=['date'])
    print(f"Loaded: {len(weather)} rows")
else:
    print("Fetching weather data from Open-Meteo API...")
    import requests

    # Split into chunks to stay within API limits
    weather_frames = []
    year_ranges = [(2001, 2005), (2006, 2010), (2011, 2015), (2016, 2020), (2021, 2025)]

    for y_start, y_end in year_ranges:
        url = (
            f"https://archive-api.open-meteo.com/v1/archive?"
            f"latitude=41.8781&longitude=-87.6298"
            f"&start_date={y_start}-01-01&end_date={y_end}-12-31"
            f"&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
            f"precipitation_sum,wind_speed_10m_max"
            f"&temperature_unit=fahrenheit"
            f"&precipitation_unit=inch"
            f"&timezone=America/Chicago"
        )
        print(f"  Requesting {y_start}-{y_end}...")
        resp = requests.get(url, timeout=60)

        if resp.status_code != 200:
            print(f"  ERROR: status {resp.status_code}")
            print(f"  Response: {resp.text[:500]}")
            continue

        data = resp.json()
        if 'daily' not in data:
            print(f"  ERROR: no 'daily' key in response")
            print(f"  Keys: {list(data.keys())}")
            continue

        daily = data['daily']
        wf = pd.DataFrame({
            'date': pd.to_datetime(daily['time']),
            'temp_max': daily.get('temperature_2m_max'),
            'temp_min': daily.get('temperature_2m_min'),
            'temp_mean': daily.get('temperature_2m_mean'),
            'precipitation': daily.get('precipitation_sum'),
            'windspeed_max': daily.get('wind_speed_10m_max'),
        })
        weather_frames.append(wf)
        print(f"  Got {len(wf)} days")

    weather = pd.concat(weather_frames, ignore_index=True)
    weather = weather.sort_values('date').reset_index(drop=True)
    weather.to_csv(WEATHER_FILE, index=False)
    print(f"\nSaved: {WEATHER_FILE} ({len(weather)} rows)")

# 2.2 Clean and validate weather data

print("=" * 60)
print("Weather Data Validation")
print("=" * 60)

print(f"\nShape: {weather.shape}")
print(f"Date range: {weather['date'].min().date()} to {weather['date'].max().date()}")

# Check for missing dates
full_dates = pd.date_range(start=weather['date'].min(), end=weather['date'].max(), freq='D')
missing_dates = full_dates.difference(weather['date'])
print(f"Expected days: {len(full_dates)}")
print(f"Actual days:   {len(weather)}")
print(f"Missing days:  {len(missing_dates)}")

if len(missing_dates) > 0:
    print(f"  Missing date samples: {missing_dates[:5].tolist()}")

# Check for null values
print(f"\nNull values per column:")
for col in ['temp_max', 'temp_min', 'temp_mean', 'precipitation', 'windspeed_max']:
    n_null = weather[col].isna().sum()
    print(f"  {col:<15s} {n_null:>6d} ({n_null/len(weather)*100:.2f}%)")

# Fill missing weather values with interpolation
for col in ['temp_max', 'temp_min', 'temp_mean', 'precipitation', 'windspeed_max']:
    if weather[col].isna().sum() > 0:
        n_before = weather[col].isna().sum()
        weather[col] = weather[col].interpolate(method='linear')
        # Fill any remaining edge NaNs
        weather[col] = weather[col].fillna(method='ffill').fillna(method='bfill')
        n_after = weather[col].isna().sum()
        print(f"  {col}: interpolated {n_before} -> {n_after} nulls")

# Sanity check value ranges
print(f"\nValue ranges:")
print(f"  temp_max:       [{weather['temp_max'].min():.1f}, {weather['temp_max'].max():.1f}] F")
print(f"  temp_min:       [{weather['temp_min'].min():.1f}, {weather['temp_min'].max():.1f}] F")
print(f"  temp_mean:      [{weather['temp_mean'].min():.1f}, {weather['temp_mean'].max():.1f}] F")
print(f"  precipitation:  [{weather['precipitation'].min():.2f}, {weather['precipitation'].max():.2f}] in")
print(f"  windspeed_max:  [{weather['windspeed_max'].min():.1f}, {weather['windspeed_max'].max():.1f}] mph")

# Flag extreme values
n_extreme_temp = ((weather['temp_max'] > 120) | (weather['temp_min'] < -40)).sum()
n_extreme_precip = (weather['precipitation'] > 10).sum()
print(f"\n  Extreme temp days (>120F or <-40F): {n_extreme_temp}")
print(f"  Extreme precip days (>10 in): {n_extreme_precip}")

if n_extreme_temp > 0:
    print("  WARNING: extreme temperatures detected, review manually")

Fetching weather data from Open-Meteo API...
  Requesting 2001-2005...
  Got 1826 days
  Requesting 2006-2010...
  Got 1826 days
  Requesting 2011-2015...
  Got 1826 days
  Requesting 2016-2020...
  Got 1827 days
  Requesting 2021-2025...
  Got 1826 days

Saved: ./data\chicago_weather_daily.csv (9131 rows)
Weather Data Validation

Shape: (9131, 6)
Date range: 2001-01-01 to 2025-12-31
Expected days: 9131
Actual days:   9131
Missing days:  0

Null values per column:
  temp_max             0 (0.00%)
  temp_min             0 (0.00%)
  temp_mean            0 (0.00%)
  precipitation        0 (0.00%)
  windspeed_max        0 (0.00%)

Value ranges:
  temp_max:       [-8.4, 99.3] F
  temp_min:       [-26.4, 82.7] F
  temp_mean:      [-18.4, 89.2] F
  precipitation:  [0.00, 5.42] in
  windspeed_max:  [6.4, 57.4] mph

  Extreme temp days (>120F or <-40F): 0
  Extreme precip days (>10 in): 0


In [17]:
# Part 3: Socioeconomic Data

# Community-level indicators from ACS via Chicago Data Portal (kn9c-c2s2).
# These are cross-sectional (time-invariant) controls.
# 3.1 Load socioeconomic data


SOCIO_FILE = os.path.join(OUTPUT_DIR, 'chicago_socioeconomic.csv')

if os.path.exists(SOCIO_FILE):
    print(f"Socioeconomic file found: {SOCIO_FILE}")
    socio = pd.read_csv(SOCIO_FILE)
    print(f"Loaded: {len(socio)} rows")
else:
    print("Fetching socioeconomic data from Chicago Data Portal...")
    import requests

    url = ("https://data.cityofchicago.org/resource/kn9c-c2s2.json"
           "?$limit=100")
    resp = requests.get(url, timeout=30)

    if resp.status_code == 200:
        socio = pd.DataFrame(resp.json())
        socio.to_csv(SOCIO_FILE, index=False)
        print(f"Saved: {SOCIO_FILE} ({len(socio)} rows)")
    else:
        print(f"ERROR: {resp.status_code}")
        print("Download manually from:")
        print("https://data.cityofchicago.org/Health-Human-Services/"
              "Census-Data-Selected-socioeconomic-indicators-in-C/kn9c-c2s2")
        socio = None

# 3.2 Clean and validate socioeconomic data

if socio is not None:
    print("=" * 60)
    print("Socioeconomic Data Validation")
    print("=" * 60)

    print(f"\nColumns: {socio.columns.tolist()}")

    # Identify community area column
    ca_col_socio = None
    for col in socio.columns:
        if 'community_area' in col.lower() or 'ca' == col.lower():
            ca_col_socio = col
            break

    if ca_col_socio is None:
        # Try to find it by content
        for col in socio.columns:
            try:
                vals = pd.to_numeric(socio[col], errors='coerce').dropna()
                if vals.between(1, 77).all() and len(vals) >= 77:
                    ca_col_socio = col
                    break
            except:
                pass

    print(f"  CA column identified: '{ca_col_socio}'")

    # Convert numeric columns
    numeric_candidates = ['percent_of_housing_crowded', 'percent_households_below_poverty',
                          'percent_aged_16_unemployed', 'percent_aged_25_without_high_school_diploma',
                          'percent_aged_under_18_or_over_64', 'per_capita_income_',
                          'hardship_index']

    socio_clean = pd.DataFrame()
    socio_clean['community_area'] = pd.to_numeric(socio[ca_col_socio], errors='coerce')

    # Try to find and rename key columns
    for col in socio.columns:
        col_lower = col.lower()
        if 'poverty' in col_lower:
            socio_clean['poverty_rate'] = pd.to_numeric(socio[col], errors='coerce')
        elif 'unemploy' in col_lower:
            socio_clean['unemployment_rate'] = pd.to_numeric(socio[col], errors='coerce')
        elif 'per_capita' in col_lower or 'income' in col_lower:
            socio_clean['per_capita_income'] = pd.to_numeric(socio[col], errors='coerce')
        elif 'hardship' in col_lower:
            socio_clean['hardship_index'] = pd.to_numeric(socio[col], errors='coerce')
        elif 'diploma' in col_lower or 'high_school' in col_lower:
            socio_clean['no_highschool_pct'] = pd.to_numeric(socio[col], errors='coerce')
        elif 'crowded' in col_lower:
            socio_clean['crowded_housing_pct'] = pd.to_numeric(socio[col], errors='coerce')

    # Remove the "CHICAGO" summary row (community_area = NaN or 0)
    socio_clean = socio_clean[socio_clean['community_area'].between(1, 77)].copy()
    socio_clean['community_area'] = socio_clean['community_area'].astype(int)

    print(f"\n  Cleaned socioeconomic data: {len(socio_clean)} rows")
    print(f"  Columns: {socio_clean.columns.tolist()}")

    # Validate completeness
    expected_ca = set(range(1, 78))
    actual_ca = set(socio_clean['community_area'].values)
    missing_ca = expected_ca - actual_ca
    print(f"  Missing CAs: {missing_ca if missing_ca else 'None'}")

    # Check for nulls
    print(f"\n  Null values:")
    for col in socio_clean.columns:
        n = socio_clean[col].isna().sum()
        print(f"    {col:<25s} {n}")

    # Value ranges
    print(f"\n  Value ranges:")
    for col in socio_clean.columns:
        if col != 'community_area':
            print(f"    {col:<25s} [{socio_clean[col].min():.1f}, {socio_clean[col].max():.1f}]")


Socioeconomic file found: ./data\chicago_socioeconomic.csv
Loaded: 78 rows
Socioeconomic Data Validation

Columns: ['ca', 'community_area_name', 'percent_of_housing_crowded', 'percent_households_below_poverty', 'percent_aged_16_unemployed', 'percent_aged_25_without_high_school_diploma', 'percent_aged_under_18_or_over_64', 'per_capita_income_', 'hardship_index']
  CA column identified: 'ca'

  Cleaned socioeconomic data: 77 rows
  Columns: ['community_area', 'crowded_housing_pct', 'poverty_rate', 'unemployment_rate', 'no_highschool_pct', 'per_capita_income', 'hardship_index']
  Missing CAs: None

  Null values:
    community_area            0
    crowded_housing_pct       0
    poverty_rate              0
    unemployment_rate         0
    no_highschool_pct         0
    per_capita_income         0
    hardship_index            0

  Value ranges:
    crowded_housing_pct       [0.3, 15.8]
    poverty_rate              [3.3, 56.5]
    unemployment_rate         [4.7, 35.9]
    no_highscho

In [18]:
# Part 4: Temporal & COVID Indicators
# Construct calendar features and COVID treatment variables.
# Illinois stay-at-home order: March 21, 2020 (Governor Pritzker).
# Phases based on Scott & Gross (2021) and Abrams (2021).

panel['year'] = panel['date'].dt.year
panel['month'] = panel['date'].dt.month
panel['day_of_week'] = panel['date'].dt.dayofweek  # 0=Mon
panel['day_of_year'] = panel['date'].dt.dayofyear
panel['is_weekend'] = (panel['day_of_week'] >= 5).astype(int)

# COVID indicators
COVID_SAH_DATE = pd.Timestamp('2020-03-21')  # Illinois stay-at-home order
COVID_REOPEN_DATE = pd.Timestamp('2020-06-03')  # Phase 3 reopening began

panel['post_covid'] = (panel['date'] >= COVID_SAH_DATE).astype(int)
panel['covid_year'] = (panel['year'] == 2020).astype(int)

# More granular COVID phases
def covid_phase(date):
    if date < pd.Timestamp('2020-03-09'):
        return 'pre_covid'
    elif date < COVID_SAH_DATE:
        return 'state_of_emergency'  # March 9 - March 20
    elif date < COVID_REOPEN_DATE:
        return 'strict_lockdown'      # March 21 - June 2
    elif date < pd.Timestamp('2021-01-01'):
        return 'partial_reopen'       # June 3 - Dec 31, 2020
    elif date < pd.Timestamp('2021-06-11'):
        return 'continued_restrictions'  # 2021 H1
    else:
        return 'post_restrictions'

panel['covid_phase'] = panel['date'].apply(covid_phase)

phase_counts = panel['covid_phase'].value_counts()
print("COVID phase distribution:")
for phase, cnt in phase_counts.items():
    print(f"  {phase:<25s} {cnt:>10,} rows")


COVID phase distribution:
  pre_covid                    539,539 rows
  post_restrictions            128,205 rows
  partial_reopen                16,324 rows
  continued_restrictions        12,397 rows
  strict_lockdown                5,698 rows
  state_of_emergency               924 rows


In [19]:
# Part 5: LISA Cluster Labels from NB03

# Attach the LISA HH/LL/HL/LH classification to each community area.
# This provides the treatment/control group for DID analysis.

# If NB03 saved LISA results, load them. Otherwise reconstruct.
LISA_FILE = os.path.join(OUTPUT_DIR, 'lisa_clusters.csv')

if os.path.exists(LISA_FILE):
    lisa_labels = pd.read_csv(LISA_FILE)
    print(f"Loaded LISA labels: {len(lisa_labels)} rows")
else:
    print("LISA labels file not found. Reconstructing from NB03...")
    print("Running Moran Local on total crime counts...")

    import geopandas as gpd
    from libpysal.weights import Queen
    from esda.moran import Moran_Local

    gdf_ca = gpd.read_file('./data/community_areas.geojson')
    ca_num_col = [c for c in gdf_ca.columns if 'area_num' in c.lower()][0]
    gdf_ca[ca_num_col] = pd.to_numeric(gdf_ca[ca_num_col], errors='coerce')

    # Aggregate total crimes per CA
    ca_total = df.groupby('community_area').size().reset_index(name='total_crimes')
    ca_total['community_area'] = ca_total['community_area'].astype(float)

    gdf = gdf_ca.merge(ca_total, left_on=ca_num_col, right_on='community_area', how='left')
    gdf['total_crimes'] = gdf['total_crimes'].fillna(0).astype(float)

    w = Queen.from_dataframe(gdf, use_index=False)
    w.transform = 'R'

    lisa = Moran_Local(gdf['total_crimes'].values, w, permutations=9999)

    cluster_map = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
    gdf['lisa_cluster'] = 'NS'
    for i in range(len(gdf)):
        if lisa.p_sim[i] < 0.05:
            gdf.iloc[i, gdf.columns.get_loc('lisa_cluster')] = cluster_map.get(lisa.q[i], 'NS')

    lisa_labels = gdf[[ca_num_col, 'lisa_cluster']].copy()
    lisa_labels.columns = ['community_area', 'lisa_cluster']
    lisa_labels['community_area'] = lisa_labels['community_area'].astype(int)
    lisa_labels.to_csv(LISA_FILE, index=False)
    print(f"Saved: {LISA_FILE}")

print(f"\nLISA cluster distribution:")
for cluster, cnt in lisa_labels['lisa_cluster'].value_counts().items():
    print(f"  {cluster:<5s} {cnt:>3d} community areas")

# Create binary indicator for DID
lisa_labels['is_high_crime_cluster'] = (lisa_labels['lisa_cluster'] == 'HH').astype(int)


LISA labels file not found. Reconstructing from NB03...
Running Moran Local on total crime counts...
Saved: ./data\lisa_clusters.csv

LISA cluster distribution:
  NS     61 community areas
  HH     11 community areas
  LL      2 community areas
  LH      2 community areas
  HL      1 community areas


In [20]:
# Part 6: Merge All Data Sources

# 6.1 Merge weather

n_before = len(panel)
panel = panel.merge(weather, on='date', how='left')
n_after = len(panel)

print("Weather merge:")
print(f"  Before: {n_before:,}  After: {n_after:,}  Change: {n_after-n_before:,}")
assert n_before == n_after, "Row count changed during weather merge!"

# Check for unmatched dates
n_missing_temp = panel['temp_mean'].isna().sum()
print(f"  Missing temp_mean after merge: {n_missing_temp:,}")

if n_missing_temp > 0:
    missing_w_dates = panel.loc[panel['temp_mean'].isna(), 'date'].unique()
    print(f"  Unmatched dates: {len(missing_w_dates)}")
    print(f"  Samples: {missing_w_dates[:5]}")
    # Fill with interpolation
    for col in ['temp_max', 'temp_min', 'temp_mean', 'precipitation', 'windspeed_max']:
        panel[col] = panel.groupby('community_area')[col].transform(
            lambda x: x.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
        )
    n_still_missing = panel['temp_mean'].isna().sum()
    print(f"  After interpolation: {n_still_missing:,} still missing")

# 6.2 Merge socioeconomic

if socio is not None and 'socio_clean' in dir():
    n_before = len(panel)
    panel = panel.merge(socio_clean, on='community_area', how='left')
    n_after = len(panel)

    print("Socioeconomic merge:")
    print(f"  Before: {n_before:,}  After: {n_after:,}  Change: {n_after-n_before:,}")
    assert n_before == n_after, "Row count changed during socio merge!"

    n_missing = panel['poverty_rate'].isna().sum()
    print(f"  Missing poverty_rate: {n_missing:,} ({n_missing/len(panel)*100:.2f}%)")

# 6.3 Merge LISA labels

n_before = len(panel)
panel = panel.merge(lisa_labels[['community_area', 'lisa_cluster', 'is_high_crime_cluster']],
                    on='community_area', how='left')
n_after = len(panel)

print("LISA merge:")
print(f"  Before: {n_before:,}  After: {n_after:,}  Change: {n_after-n_before:,}")
assert n_before == n_after, "Row count changed during LISA merge!"

n_missing = panel['lisa_cluster'].isna().sum()
print(f"  Missing LISA label: {n_missing:,}")

Weather merge:
  Before: 703,087  After: 703,087  Change: 0
  Missing temp_mean after merge: 0
Socioeconomic merge:
  Before: 703,087  After: 703,087  Change: 0
  Missing poverty_rate: 0 (0.00%)
LISA merge:
  Before: 703,087  After: 703,087  Change: 0
  Missing LISA label: 0


In [21]:
# Part 7: Lag Features

print("Computing lag features (per community area)...")
panel = panel.sort_values(['community_area', 'date']).reset_index(drop=True)

# Lag features
for lag in [1, 7, 14, 30]:
    col_name = f'crime_lag_{lag}d'
    panel[col_name] = panel.groupby('community_area')['crime_total'].shift(lag)
    n_null = panel[col_name].isna().sum()
    print(f"  {col_name}: {n_null:,} nulls (expected: {lag * len(all_ca):,})")

# Rolling averages
for window in [7, 30]:
    col_name = f'crime_rolling_{window}d'
    panel[col_name] = (panel.groupby('community_area')['crime_total']
                       .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()))
    n_null = panel[col_name].isna().sum()
    print(f"  {col_name}: {n_null:,} nulls")

# City-wide daily average (spatial context)
city_daily = panel.groupby('date')['crime_total'].mean().reset_index(name='city_avg_crime')
panel = panel.merge(city_daily, on='date', how='left')

# Neighbor average from LISA spatial structure would go here in NB05

print("Lag features complete.")


Computing lag features (per community area)...
  crime_lag_1d: 77 nulls (expected: 78)
  crime_lag_7d: 539 nulls (expected: 546)
  crime_lag_14d: 1,078 nulls (expected: 1,092)
  crime_lag_30d: 2,310 nulls (expected: 2,340)
  crime_rolling_7d: 77 nulls
  crime_rolling_30d: 77 nulls
Lag features complete.


In [24]:
# Part 8: Final Validation

# Comprehensive checks before export.

print("=" * 60)
print("FINAL PANEL VALIDATION")
print("=" * 60)

print(f"\nShape: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
print(f"Date range: {panel['date'].min().date()} to {panel['date'].max().date()}")
print(f"Community areas: {panel['community_area'].nunique()}")
print(f"Years: {panel['year'].min()} - {panel['year'].max()}")

# Check completeness
n_ca = panel['community_area'].nunique()
n_days = panel['date'].nunique()
expected = n_ca * n_days
print(f"\nExpected rows: {n_ca} CA x {n_days} days = {expected:,}")
print(f"Actual rows:   {panel.shape[0]:,}")
assert panel.shape[0] == expected, "INCOMPLETE PANEL!"
print("PASS: Complete panel.")

# Check no duplicate rows
n_dup = panel.duplicated(subset=['community_area', 'date']).sum()
print(f"\nDuplicate (CA, date) pairs: {n_dup}")
assert n_dup == 0, "DUPLICATE ROWS!"
print("PASS: No duplicates.")

# Check all Y columns are non-negative integers
print(f"\nY variable checks:")
for col in y_cols:
    vmin = panel[col].min()
    has_null = panel[col].isna().any()
    is_int = (panel[col] == panel[col].astype(int)).all()
    status = "PASS" if (vmin >= 0 and not has_null and is_int) else "FAIL"
    print(f"  {col:<20s} min={vmin:>4d}  null={has_null}  int={is_int}  [{status}]")

# Column inventory
print(f"\nFull column list ({len(panel.columns)}):")
for i, col in enumerate(panel.columns, 1):
    dtype = panel[col].dtype
    n_null = panel[col].isna().sum()
    null_str = f"{n_null:,} nulls" if n_null > 0 else "OK"
    print(f"  {i:>2d}. {col:<30s} {str(dtype):<15s} {null_str}")


FINAL PANEL VALIDATION

Shape: 703,087 rows x 44 cols
Date range: 2001-01-01 to 2025-12-31
Community areas: 77
Years: 2001 - 2025

Expected rows: 77 CA x 9131 days = 703,087
Actual rows:   703,087
PASS: Complete panel.

Duplicate (CA, date) pairs: 0
PASS: No duplicates.

Y variable checks:
  crime_total          min=   0  null=False  int=True  [PASS]
  crime_violent        min=   0  null=False  int=True  [PASS]
  crime_property       min=   0  null=False  int=True  [PASS]
  crime_theft          min=   0  null=False  int=True  [PASS]
  crime_battery        min=   0  null=False  int=True  [PASS]
  crime_homicide       min=   0  null=False  int=True  [PASS]
  crime_burglary       min=   0  null=False  int=True  [PASS]
  crime_mvt            min=   0  null=False  int=True  [PASS]
  crime_narcotics      min=   0  null=False  int=True  [PASS]
  crime_robbery        min=   0  null=False  int=True  [PASS]
  crime_assault        min=   0  null=False  int=True  [PASS]
  arrest_count         min=

In [25]:
# Part 9: Export

PANEL_FILE = os.path.join(OUTPUT_DIR, 'panel_ca_daily.parquet')
panel.to_parquet(PANEL_FILE, index=False, engine='pyarrow')
size_mb = os.path.getsize(PANEL_FILE) / (1024**2)

print(f"\nExported: {PANEL_FILE}")
print(f"  Size:  {size_mb:.0f} MB")
print(f"  Rows:  {panel.shape[0]:,}")
print(f"  Cols:  {panel.shape[1]}")

# Quick summary of key columns
print(f"\nPanel summary:")
print(f"  Y variables: {len(y_cols) + 1} (including arrest_rate)")
print(f"  Weather vars: 5 (temp_max/min/mean, precipitation, windspeed)")
print(f"  Socio vars: {len(socio_clean.columns)-1 if 'socio_clean' in dir() else 'N/A'}")
print(f"  Lag features: 6 (1d/7d/14d/30d lags + 7d/30d rolling)")
print(f"  COVID indicators: 3 (post_covid, covid_year, covid_phase)")
print(f"  LISA label: lisa_cluster, is_high_crime_cluster")

panel.head(3)


Exported: ./data\panel_ca_daily.parquet
  Size:  15 MB
  Rows:  703,087
  Cols:  44

Panel summary:
  Y variables: 14 (including arrest_rate)
  Weather vars: 5 (temp_max/min/mean, precipitation, windspeed)
  Socio vars: 6
  Lag features: 6 (1d/7d/14d/30d lags + 7d/30d rolling)
  COVID indicators: 3 (post_covid, covid_year, covid_phase)
  LISA label: lisa_cluster, is_high_crime_cluster


,community_area,date,crime_total,crime_violent,crime_property,crime_theft,crime_battery,crime_homicide,crime_burglary,crime_mvt,...,hardship_index,lisa_cluster,is_high_crime_cluster,crime_lag_1d,crime_lag_7d,crime_lag_14d,crime_lag_30d,crime_rolling_7d,crime_rolling_30d,city_avg_crime
0,1,2001-01-01,7,0,2,2,0,0,0,0,...,39.0,NS,0,NaN,NaN,NaN,NaN,NaN,NaN,3.662338
1,1,2001-01-02,0,0,0,0,0,0,0,0,...,39.0,NS,0,7.0,NaN,NaN,NaN,7.0,7.0,0.220779
2,1,2001-01-03,0,0,0,0,0,0,0,0,...,39.0,NS,0,0.0,NaN,NaN,NaN,3.5,3.5,0.220779
